# Phần 1 — Câu hỏi Trắc nghiệm
Chạy từng cell, ghi đáp án vào form nộp bài.

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DATA = '../data/'

orders      = pd.read_csv(DATA+'orders.csv',      parse_dates=['order_date'])
order_items = pd.read_csv(DATA+'order_items.csv', low_memory=False)
customers   = pd.read_csv(DATA+'customers.csv',   parse_dates=['signup_date'])
products    = pd.read_csv(DATA+'products.csv')
payments    = pd.read_csv(DATA+'payments.csv')
returns     = pd.read_csv(DATA+'returns.csv',     parse_dates=['return_date'])
geography   = pd.read_csv(DATA+'geography.csv')
web_traffic = pd.read_csv(DATA+'web_traffic.csv', parse_dates=['date'])
sales       = pd.read_csv(DATA+'sales.csv',       parse_dates=['Date'])
print('Data loaded OK')


Data loaded OK


## Q1 — Median inter-order gap


In [3]:
# Khách hàng có đơn hoàn thành → tính khoảng cách giữa 2 đơn liên tiếp → median
o = orders[orders['order_status'] == 'delivered'].sort_values(['customer_id','order_date'])
o['prev_date'] = o.groupby('customer_id')['order_date'].shift(1)
o['gap_days']  = (o['order_date'] - o['prev_date']).dt.days
multi = o[o['prev_date'].notna()]
median_gap = multi['gap_days'].median()
print(f'Median inter-order gap: {median_gap:.1f} ngày')

options_q1 = {'A': 30, 'B': 90, 'C': 180, 'D': 365}
ans_q1 = min(options_q1, key=lambda k: abs(options_q1[k] - median_gap))
print(f'\nĐáp án: {ans_q1}')

Median inter-order gap: 175.0 ngày

Đáp án: C


## Q2 — Segment có gross margin trung bình cao nhất


In [4]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
result = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(result)

options_q2 = {'A': 'Premium', 'B': 'Performance', 'C': 'Activewear', 'D': 'Standard'}
best_val = result.idxmax()
ans_q2 = next((k for k, v in options_q2.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q2}')

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

Đáp án: D


## Q3 — Return reason phổ biến nhất trong Streetwear


In [5]:
ret_prod = returns.merge(products[['product_id','category']], on='product_id')
streetwear = ret_prod[ret_prod['category'] == 'Streetwear']
print(streetwear['return_reason'].value_counts())

options_q3 = {'A': 'defective', 'B': 'wrong_size', 'C': 'changed_mind', 'D': 'not_as_described'}
best_val = streetwear['return_reason'].value_counts().idxmax()
ans_q3 = next((k for k, v in options_q3.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q3}')

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Đáp án: B


## Q4 — Traffic source có bounce rate thấp nhất


In [6]:
result = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(result)

options_q4 = {'A': 'organic_search', 'B': 'paid_search', 'C': 'email_campaign', 'D': 'social_media'}
best_val = result.idxmin()
ans_q4 = next((k for k, v in options_q4.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q4}')

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Đáp án: C


## Q5 — % order_items có promo_id không null


In [7]:
pct = order_items['promo_id'].notna().mean() * 100
print(f'Tỷ lệ có promo_id: {pct:.1f}%')

options_q5 = {'A': 12, 'B': 25, 'C': 39, 'D': 54}
ans_q5 = min(options_q5, key=lambda k: abs(options_q5[k] - pct))
print(f'\nĐáp án: {ans_q5}')

Tỷ lệ có promo_id: 38.7%

Đáp án: C


## Q6 — Age group có số đơn hàng trung bình cao nhất


In [8]:
cust_orders = orders.groupby('customer_id').size().reset_index(name='order_count')
merged = customers[customers['age_group'].notna()].merge(cust_orders, on='customer_id', how='left')
merged['order_count'] = merged['order_count'].fillna(0)
result = merged.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print(result)

options_q6 = {'A': '55+', 'B': '25-34', 'C': '35-44', 'D': '45-54'}
best_val = result.idxmax()
ans_q6 = next((k for k, v in options_q6.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q6}')

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64

Đáp án: A


## Q7 — Region có tổng doanh thu cao nhất (join orders → geography)


In [9]:
# sales.csv không có region → join orders → geography → tổng theo ngày
orders_geo = orders.merge(geography[['zip','region']].drop_duplicates('zip'), on='zip', how='left')
orders_geo['order_date_str'] = orders_geo['order_date'].dt.strftime('%Y-%m-%d')
sales['date_str'] = sales['Date'].dt.strftime('%Y-%m-%d')

daily_region = orders_geo.groupby(['order_date_str','region']).size().reset_index(name='n_orders')
daily_total  = daily_region.groupby('order_date_str')['n_orders'].transform('sum')
daily_region['rev_share'] = daily_region['n_orders'] / daily_total

daily_region = daily_region.merge(sales[['date_str','Revenue']], left_on='order_date_str', right_on='date_str', how='left')
daily_region['rev_region'] = daily_region['rev_share'] * daily_region['Revenue']

result = daily_region.groupby('region')['rev_region'].sum().sort_values(ascending=False)
print(result)

options_q7 = {'A': 'West', 'B': 'Central', 'C': 'East', 'D': 'Cả ba vùng'}
best_val = result.idxmax()
ans_q7 = next((k for k, v in options_q7.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q7}')

region
East       7.480137e+09
Central    4.730484e+09
West       4.219855e+09
Name: rev_region, dtype: float64

Đáp án: C


## Q8 — Payment method phổ biến nhất trong đơn cancelled


In [10]:
cancelled = orders[orders['order_status'] == 'cancelled']
result = cancelled['payment_method'].value_counts()
print(result)

options_q8 = {'A': 'credit_card', 'B': 'cod', 'C': 'paypal', 'D': 'bank_transfer'}
best_val = result.idxmax()
ans_q8 = next((k for k, v in options_q8.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q8}')

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

Đáp án: A


## Q9 — Size có return rate cao nhất


In [11]:
# return rate = số return records / số order_item records, theo size
oi_size = order_items.merge(products[['product_id','size']], on='product_id')
ret_size = returns.merge(products[['product_id','size']], on='product_id')

oi_count  = oi_size[oi_size['size'].isin(['S','M','L','XL'])].groupby('size').size()
ret_count = ret_size[ret_size['size'].isin(['S','M','L','XL'])].groupby('size').size()
rate = (ret_count / oi_count).sort_values(ascending=False)
print(rate)

options_q9 = {'A': 'S', 'B': 'M', 'C': 'L', 'D': 'XL'}
best_val = rate.idxmax()
ans_q9 = next((k for k, v in options_q9.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q9}')

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64

Đáp án: A


## Q10 — Installment plan có payment_value trung bình cao nhất


In [12]:
result = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(result)

options_q10 = {'A': 1, 'B': 3, 'C': 6, 'D': 12}
best_val = result.idxmax()
ans_q10 = next((k for k, v in options_q10.items() if v == best_val), '???')
print(f'\nĐáp án: {ans_q10}')

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

Đáp án: C


## Tổng hợp đáp án

In [13]:
import pandas as pd
from IPython.display import display

print('=== Tong hop dap an MCQ ===')
print()
try:
    answers = {
        'Q1':  ans_q1,  'Q2':  ans_q2,  'Q3':  ans_q3,
        'Q4':  ans_q4,  'Q5':  ans_q5,  'Q6':  ans_q6,
        'Q7':  ans_q7,  'Q8':  ans_q8,  'Q9':  ans_q9,
        'Q10': ans_q10,
    }
    df_ans = pd.DataFrame(
        list(answers.items()),
        columns=['Cau hoi', 'Dap an']
    ).set_index('Cau hoi')
    display(df_ans)
    print()
    chuoi = ', '.join(f'{k}: {v}' for k, v in answers.items())
    print(f'Chuoi dap an: {chuoi}')
    print()
    print('Submit string:', ' '.join(v for v in answers.values()))
except NameError as e:
    print(f'[!] Vui long chay tat ca cac cell phia tren truoc! ({e})')


=== Tong hop dap an MCQ ===



,Dap an
Cau hoi,
Q1,C
Q2,D
Q3,B
Q4,C
Q5,C
Q6,A
Q7,C
Q8,A
Q9,A



Chuoi dap an: Q1: C, Q2: D, Q3: B, Q4: C, Q5: C, Q6: A, Q7: C, Q8: A, Q9: A, Q10: C

Submit string: C D B C C A C A A C
